In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
List @Aitechskill_Stage

In [ ]:
SELECT COUNT(*)
FROM DIRECTORY(@AITECHSKILL_DB.AITECHSKILL_SCHEMA.AITECHSKILL_STAGE);

In [ ]:
LIST @AITECHSKILL_DB.AITECHSKILL_SCHEMA.AITECHSKILL_STAGE;

SELECT COUNT(*)
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));


In [ ]:
CREATE OR REPLACE TABLE raw_customer_reviews AS
SELECT
    RELATIVE_PATH as filename,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            '@AITECHSKILL_STAGE',
            RELATIVE_PATH
    ) AS extracted_content
FROM DIRECTORY('@AITECHSKILL_STAGE')
WHERE RELATIVE_PATH LIKE '%.docx';

In [ ]:
SELECT 
    filename,
    extracted_content
FROM raw_customer_reviews
LIMIT 3;


In [ ]:
SELECT
    filename,
    
    REGEXP_SUBSTR(
        extracted_content:content,
        'Product: (.*?)\\n',
        1, 1, 'e'
    ) AS product,

    TO_DATE(
        REGEXP_SUBSTR(
            extracted_content:content,
            'Date: (202[0-9]-[0-9]{2}-[0-9]{2})',
            1, 1, 'e'
        )
    ) AS review_date,

    CASE
        WHEN POSITION('Customer Review' IN extracted_content:content) > 0 THEN
            SUBSTRING(
                extracted_content:content,
                POSITION('Customer Review' IN extracted_content:content)
                + LENGTH('Customer Review')
            )
        ELSE NULL
    END AS review_text,

    CAST(
        CONCAT('2', REGEXP_SUBSTR(filename, '\\d+'))
        AS INTEGER
    ) AS order_id

FROM raw_customer_reviews;


In [ ]:
CREATE OR REPLACE TABLE aitechskill_db.aitechskill_schema.customer_reviews (
    filename VARCHAR,
    product VARCHAR,
    review_date DATE,
    review_text VARCHAR,
    order_id INTEGER
);

In [ ]:
SELECT * FROM {{cell8}} LIMIT 5;


In [ ]:
INSERT INTO aitechskill_db.aitechskill_schema.customer_reviews 
    (filename, product, review_date, review_text, order_id)
SELECT * FROM {{cell8}};

In [ ]:
SELECT * FROM {{cell8}} LIMIT 5;

In [ ]:
DESC TABLE CUSTOMER_REVIEWS;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

shipping_df = (
    session.read.options({
        "FIELD_DELIMITER": ",",
        "SKIP_HEADER": 1
    })
    .csv("@AITECHSKILL_STAGE/shipping_logs.csv")
)

shipping_df.show()

In [ ]:
shipping_df = shipping_df.select(
    col('"c1"').alias("order_id"),
    col('"c2"').alias("shipping_date"),
    col('"c3"').alias("carrier"),
    col('"c4"').alias("tracking_number"),
    col('"c5"').alias("latitude"),
    col('"c6"').alias("longitude"),
    col('"c7"').alias("status"),
    col('"c8"').alias("delivery_days"),
    col('"c9"').alias("late"),
    col('"c10"').alias("region")
)


In [ ]:
shipping_df.write.save_as_table("shipping_logs", mode="overwrite")


In [ ]:
pandas_df = shipping_df.to_pandas()
pandas_df.describe()
pandas_df.head(10)


In [ ]:
DESC TABLE SHIPPING_LOGS;

In [ ]:
CREATE OR REPLACE TABLE merged_reviews AS
SELECT 
  r.order_id,
  r.filename,
  r.product,
  r.review_date,
  r.review_text,
  s.shipping_date,
  s.carrier,
  s.tracking_number,
  s.latitude,
  s.longitude,
  s.status,
  s.delivery_days,
  s.late,
  s.region
FROM 
  customer_reviews r
JOIN 
  shipping_logs s
ON 
  r.order_id = s.order_id;

In [ ]:
SELECT * FROM merged_reviews LIMIT 10;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
# Load parsed reviews into a DataFrame
snowpark_df = session.sql("SELECT * FROM merged_reviews")

# Show a sample of the data
snowpark_df.show()

In [ ]:
DESC TABLE merged_reviews;

In [ ]:
from snowflake.snowpark.functions import lower, trim, col

# Standardize the review text
df_lowercase = snowpark_df.with_column("review_text", trim(lower(col("review_text"))))

In [ ]:
df_lowercase.write.mode("overwrite").save_as_table("clean_reviews")

In [ ]:
DESC TABLE clean_reviews;


In [ ]:
query = """
SELECT 
    *,
    SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) as SENTIMENT_SCORE
FROM 
    CLEAN_REVIEWS
"""

df = session.sql(query).to_pandas()

df.head()

In [ ]:
import matplotlib 
print(matplotlib.__version__)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.hist(df["SENTIMENT_SCORE"], bins=20, color='skyblue', edgecolor='black')

plt.title("Histogram of Sentiment Scores")
plt.xlabel("Sentiment Score")
plt.ylabel("Number of Reviews")

plt.grid(axis='y', alpha=0.7)
plt.show()

In [ ]:
SELECT 
    ORDER_ID,
    PRODUCT,
    REVIEW_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) AS sentiment_score,
    CASE 
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) >= 0.7 THEN 'Very Positive'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) >= 0.3 THEN 'Positive'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) >= -0.3 THEN 'Neutral'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) >= -0.7 THEN 'Negative'
        ELSE 'Very Negative'
    END AS sentiment_label
FROM 
    clean_reviews;

In [ ]:
# Save the data to a new table
# df.write.save_as_table("reviews_with_sentiment", mode="overwrite")
from snowflake.snowpark import Session

# assuming you already have a session object
snow_df = session.create_dataframe(df)

snow_df.write.save_as_table(
    "reviews_with_sentiment",
    mode="overwrite"
)


In [ ]:
query = """
SELECT
    *,
FROM 
    REVIEWS_WITH_SENTIMENT
"""

df = session.sql(query).to_pandas()

df.head()

In [ ]:
import matplotlib.pyplot as plt

product_sentiment = df.groupby("PRODUCT")["SENTIMENT_SCORE"].mean().sort_values()
product_sentiment.plot(kind="barh", title="Average Sentiment by Product")
plt.xlabel("Sentiment Score")
plt.tight_layout()
plt.show()

In [ ]:
shipment_counts = (
    df.groupby('SHIPPING_DATE')['ORDER_ID']
      .count()
      .reset_index(name='SHIPMENT_COUNT')
      .sort_values('SHIPPING_DATE')
)

shipment_counts.head()

In [ ]:
shipment_counts.plot(x="SHIPPING_DATE", y="SHIPMENT_COUNT", kind="line", title="Shipments Per Day")
plt.ylabel("Number of Shipments")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
low_volume_days = shipment_counts[shipment_counts["SHIPMENT_COUNT"] < 5]
print("Low-volume shipping days:\n", low_volume_days)

In [ ]:
carrier_counts = df.groupby("CARRIER")["ORDER_ID"].count()
carrier_counts.plot(kind="bar", title="Total Shipments by Carrier")
plt.ylabel("Number of Shipments")
plt.tight_layout()
plt.show()

In [ ]:
avg_sentiment = df.groupby("STATUS")["SENTIMENT_SCORE"].mean().sort_values()

avg_sentiment.plot(kind="barh", title="Avg Sentiment by Shipping Status")
plt.xlabel("Sentiment Score")
plt.ylabel("Shipping Status")
plt.tight_layout()
plt.show()

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.cortex import complete

# Initialize the Streamlit app
st.title("Avalanche Streamlit App")

# Get data from Snowflake
session = get_active_session()
query = """
SELECT
    *
FROM
    REVIEWS_WITH_SENTIMENT
"""
df_reviews = session.sql(query).to_pandas()
df_string = df_reviews.to_string(index=False)

# Convert date columns to datetime
df_reviews['REVIEW_DATE'] = pd.to_datetime(df_reviews['REVIEW_DATE'])
df_reviews['SHIPPING_DATE'] = pd.to_datetime(df_reviews['SHIPPING_DATE'])

# Visualization: Average Sentiment by Product
st.subheader("Average Sentiment by Product")
product_sentiment = df_reviews.groupby("PRODUCT")["SENTIMENT_SCORE"].mean().sort_values()

fig, ax = plt.subplots()
product_sentiment.plot(kind="barh", ax=ax, title="Average Sentiment by Product")
ax.set_xlabel("Sentiment Score")
plt.tight_layout()
st.pyplot(fig)

# Product filter on the main page
st.subheader("Filter by Product")

product = st.selectbox("Choose a product", ["All Products"] + list(df_reviews["PRODUCT"].unique()))

if product != "All Products":
    filtered_data = df_reviews[df_reviews["PRODUCT"] == product]
else:
    filtered_data = df_reviews


# Display the filtered data as a table
st.subheader(f"📁 Reviews for {product}")
st.dataframe(filtered_data)

# Visualization: Sentiment Distribution for Selected Products
st.subheader(f"Sentiment Distribution for {product}")
fig, ax = plt.subplots()
filtered_data['SENTIMENT_SCORE'].hist(ax=ax, bins=20)
ax.set_title("Distribution of Sentiment Scores")
ax.set_xlabel("Sentiment Score")
ax.set_ylabel("Frequency")
st.pyplot(fig)

# Chatbot for Q&A
st.subheader("Ask Questions About Your Data")
user_question = st.text_input("Enter your question here:")

if user_question:

    response = complete(model="claude-3-5-sonnet", prompt=f"Answer this question using the dataset: {user_question} <context>{df_string}</context>", session=session)
    st.write(response)

In [ ]:
"""
Snowpark Python Pipeline for Sentiment Analysis on CLEAN_REVIEWS Table
Analyzes customer reviews with delivery performance correlation

Usage in Snowflake Notebook:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    
    # Run full pipeline
    results = run_full_pipeline(session)
    display_summary(results)
"""

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, when, avg, count, sum as sum_, round as round_,
    date_trunc, current_timestamp, corr, stddev, year,
    call_builtin, lit, min as min_, max as max_
)
from snowflake.snowpark.types import StringType, FloatType, IntegerType

# Get active session (for Snowflake Notebooks)
session = get_active_session()


# ============================================================================
# 2. SENTIMENT ENRICHMENT PIPELINE
# ============================================================================

def enrich_reviews_with_sentiment(session, source_table="clean_reviews", 
                                   target_table="clean_reviews_enriched"):
    """
    Read clean_reviews and enrich with Cortex sentiment analysis
    """
    print(f"Loading data from {source_table}...")
    reviews_df = session.table(source_table).filter(col("review_text").is_not_null())
    
    print("Applying sentiment analysis...")
    # Apply sentiment analysis using Cortex
    enriched_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    )
    
    # Add basic sentiment label
    enriched_df = enriched_df.with_column(
        "sentiment_label",
        when(col("sentiment_score") >= 0.5, lit("Positive"))
        .when(col("sentiment_score") <= -0.5, lit("Negative"))
        .otherwise(lit("Neutral"))
    )
    
    # Add detailed sentiment label
    enriched_df = enriched_df.with_column(
        "sentiment_label_detailed",
        when(col("sentiment_score") >= 0.7, lit("Very Positive"))
        .when(col("sentiment_score") >= 0.3, lit("Positive"))
        .when(col("sentiment_score") >= -0.3, lit("Neutral"))
        .when(col("sentiment_score") >= -0.7, lit("Negative"))
        .otherwise(lit("Very Negative"))
    )
    
    # Add enrichment timestamp
    enriched_df = enriched_df.with_column(
        "enriched_at",
        current_timestamp()
    )
    
    # Write to target table
    print(f"Writing enriched data to {target_table}...")
    enriched_df.write.mode("overwrite").save_as_table(target_table)
    
    record_count = enriched_df.count()
    print(f"✓ Enriched {record_count} reviews with sentiment scores")
    
    return enriched_df


# ============================================================================
# 3. SENTIMENT ANALYSIS BY PRODUCT
# ============================================================================

def analyze_sentiment_by_product(session, source_table="clean_reviews"):
    """
    Aggregate sentiment metrics by product
    """
    print("Analyzing sentiment by product...")
    reviews_df = session.table(source_table).filter(col("review_text").is_not_null())
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    )
    
    # Aggregate by product
    product_sentiment = analysis_df.group_by(
        col("product")
    ).agg(
        count(col("order_id")).alias("total_reviews"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment_score"),
        sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)).alias("positive_count"),
        sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)).alias("negative_count"),
        round_(
            (sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("positive_pct"),
        round_(
            (sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("negative_pct"),
        sum_(when(col("late") == True, 1).otherwise(0)).alias("late_deliveries"),
        round_(
            (sum_(when(col("late") == True, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("late_pct")
    ).sort(col("avg_sentiment_score").desc())
    
    print(f"✓ Analyzed {product_sentiment.count()} products")
    return product_sentiment


# ============================================================================
# 4. SENTIMENT ANALYSIS BY REGION
# ============================================================================

def analyze_sentiment_by_region(session, source_table="clean_reviews"):
    """
    Aggregate sentiment metrics by region
    """
    print("Analyzing sentiment by region...")
    reviews_df = session.table(source_table).filter(col("review_text").is_not_null())
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    )
    
    # Aggregate by region
    region_sentiment = analysis_df.group_by(
        col("region")
    ).agg(
        count(col("order_id")).alias("total_reviews"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment_score"),
        sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)).alias("positive_count"),
        sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)).alias("negative_count"),
        round_(
            (sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("positive_pct"),
        round_(avg(col("delivery_days")), 1).alias("avg_delivery_days"),
        round_(
            (sum_(when(col("late") == True, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("late_pct")
    ).sort(col("avg_sentiment_score").desc())
    
    print(f"✓ Analyzed {region_sentiment.count()} regions")
    return region_sentiment


# ============================================================================
# 5. SENTIMENT ANALYSIS BY CARRIER
# ============================================================================

def analyze_sentiment_by_carrier(session, source_table="clean_reviews"):
    """
    Aggregate sentiment metrics by carrier
    """
    print("Analyzing sentiment by carrier...")
    reviews_df = session.table(source_table).filter(
        (col("review_text").is_not_null()) & (col("carrier").is_not_null())
    )
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    )
    
    # Aggregate by carrier
    carrier_sentiment = analysis_df.group_by(
        col("carrier")
    ).agg(
        count(col("order_id")).alias("total_reviews"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment_score"),
        round_(
            (sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("positive_pct"),
        round_(
            (sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("negative_pct"),
        round_(avg(col("delivery_days")), 1).alias("avg_delivery_days"),
        round_(
            (sum_(when(col("late") == True, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("late_delivery_pct")
    ).sort(col("avg_sentiment_score").desc())
    
    print(f"✓ Analyzed {carrier_sentiment.count()} carriers")
    return carrier_sentiment


# ============================================================================
# 6. DELIVERY DELAY AND SENTIMENT CORRELATION
# ============================================================================

def correlate_sentiment_with_delays(session, source_table="clean_reviews"):
    """
    Analyze correlation between delivery delays and sentiment
    """
    print("Analyzing delivery delay correlation...")
    reviews_df = session.table(source_table).filter(
        (col("review_text").is_not_null()) & (col("delivery_days").is_not_null())
    )
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    ).with_column(
        "is_late",
        when(col("late") == True, 1).otherwise(0)
    )
    
    # Calculate correlation statistics
    correlation_stats = analysis_df.agg([
        corr(col("sentiment_score"), col("delivery_days")).alias("sentiment_delivery_correlation"),
        avg(when(col("is_late") == 1, col("sentiment_score"))).alias("avg_sentiment_late"),
        avg(when(col("is_late") == 0, col("sentiment_score"))).alias("avg_sentiment_ontime"),
        sum_(when(col("is_late") == 1, 1).otherwise(0)).alias("late_count"),
        sum_(when(col("is_late") == 0, 1).otherwise(0)).alias("ontime_count"),
        round_(avg(when(col("is_late") == 1, col("delivery_days"))), 1).alias("avg_days_late"),
        round_(avg(when(col("is_late") == 0, col("delivery_days"))), 1).alias("avg_days_ontime")
    ])
    
    # Calculate sentiment difference
    stats_row = correlation_stats.collect()[0]
    
    print("✓ Correlation analysis complete")
    return correlation_stats, analysis_df


# ============================================================================
# 7. ANALYZE BY DELIVERY TIME BUCKETS
# ============================================================================

def analyze_by_delivery_time(session, source_table="clean_reviews"):
    """
    Break down sentiment by delivery time buckets
    """
    print("Analyzing by delivery time buckets...")
    reviews_df = session.table(source_table).filter(
        (col("review_text").is_not_null()) & (col("delivery_days").is_not_null())
    )
    
    # Calculate sentiment and create buckets
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    ).with_column(
        "delivery_time_bucket",
        when(col("delivery_days") <= 2, lit("0-2 Days"))
        .when(col("delivery_days") <= 4, lit("3-4 Days"))
        .when(col("delivery_days") <= 6, lit("5-6 Days"))
        .when(col("delivery_days") <= 8, lit("7-8 Days"))
        .otherwise(lit("9+ Days"))
    )
    
    # Aggregate by bucket
    time_analysis = analysis_df.group_by(
        col("delivery_time_bucket")
    ).agg(
        count(col("order_id")).alias("review_count"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment"),
        round_(stddev(col("sentiment_score")), 3).alias("sentiment_stddev"),
        round_(
            (sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("negative_pct"),
        round_(
            (sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("positive_pct"),
        round_(
            (sum_(when(col("late") == True, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("late_pct")
    ).sort(col("delivery_time_bucket"))
    
    print(f"✓ Created {time_analysis.count()} delivery time buckets")
    return time_analysis


# ============================================================================
# 8. IDENTIFY PROBLEM AREAS
# ============================================================================

def identify_problem_areas(session, source_table="clean_reviews", min_sample_size=5):
    """
    Identify products/regions/carriers with worst sentiment when late
    """
    print("Identifying problem areas...")
    reviews_df = session.table(source_table).filter(
        (col("review_text").is_not_null()) & (col("late") == True)
    )
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    )
    
    # Aggregate by product, region, carrier
    problem_areas = analysis_df.group_by(
        col("product"),
        col("region"),
        col("carrier")
    ).agg(
        count(col("order_id")).alias("total_late_reviews"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment_when_late"),
        sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)).alias("negative_reviews"),
        round_(
            (sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("negative_pct"),
        round_(avg(col("delivery_days")), 1).alias("avg_delivery_days")
    ).filter(
        col("total_late_reviews") >= min_sample_size
    ).sort(
        col("negative_pct").desc(),
        col("avg_sentiment_when_late").asc()
    )
    
    print(f"✓ Identified {problem_areas.count()} problem areas")
    return problem_areas


# ============================================================================
# 9. CREATE DASHBOARD VIEW
# ============================================================================

def create_dashboard_view(session, source_table="clean_reviews", 
                          view_name="vw_sentiment_dashboard"):
    """
    Create a comprehensive view for dashboard consumption
    """
    print("Creating dashboard view...")
    reviews_df = session.table(source_table).filter(col("review_text").is_not_null())
    
    # Add calculated fields
    dashboard_df = reviews_df.with_column(
        "review_month", date_trunc("month", col("review_date"))
    ).with_column(
        "review_week", date_trunc("week", col("review_date"))
    ).with_column(
        "review_year", year(col("review_date"))
    ).with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    ).with_column(
        "sentiment_label",
        when(col("sentiment_score") >= 0.5, lit("Positive"))
        .when(col("sentiment_score") <= -0.5, lit("Negative"))
        .otherwise(lit("Neutral"))
    ).with_column(
        "sentiment_label_detailed",
        when(col("sentiment_score") >= 0.7, lit("Very Positive"))
        .when(col("sentiment_score") >= 0.3, lit("Positive"))
        .when(col("sentiment_score") >= -0.3, lit("Neutral"))
        .when(col("sentiment_score") >= -0.7, lit("Negative"))
        .otherwise(lit("Very Negative"))
    ).with_column(
        "delivery_status",
        when(col("late") == True, lit("Late")).otherwise(lit("On-Time"))
    ).with_column(
        "delivery_speed",
        when(col("delivery_days") <= 2, lit("Fast (0-2 days)"))
        .when(col("delivery_days") <= 4, lit("Normal (3-4 days)"))
        .when(col("delivery_days") <= 6, lit("Slow (5-6 days)"))
        .otherwise(lit("Very Slow (7+ days)"))
    ).with_column(
        "is_positive", when(col("sentiment_score") >= 0.5, 1).otherwise(0)
    ).with_column(
        "is_negative", when(col("sentiment_score") <= -0.5, 1).otherwise(0)
    ).with_column(
        "is_late", when(col("late") == True, 1).otherwise(0)
    )
    
    # Create or replace view
    dashboard_df.create_or_replace_view(view_name)
    
    print(f"✓ Created view: {view_name}")
    return dashboard_df


# ============================================================================
# 10. TIME SERIES ANALYSIS
# ============================================================================

def analyze_sentiment_trends(session, source_table="clean_reviews"):
    """
    Analyze sentiment trends over time
    """
    print("Analyzing sentiment trends over time...")
    reviews_df = session.table(source_table).filter(col("review_text").is_not_null())
    
    # Calculate sentiment
    analysis_df = reviews_df.with_column(
        "sentiment_score",
        call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", col("review_text"))
    ).with_column(
        "month", date_trunc("month", col("review_date"))
    )
    
    # Aggregate by month
    trends = analysis_df.group_by(
        col("month")
    ).agg(
        count(col("order_id")).alias("total_reviews"),
        round_(avg(col("sentiment_score")), 3).alias("avg_sentiment"),
        round_(
            (sum_(when(col("sentiment_score") >= 0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("positive_pct"),
        round_(
            (sum_(when(col("sentiment_score") <= -0.5, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("negative_pct"),
        round_(
            (sum_(when(col("late") == True, 1).otherwise(0)) / count(col("order_id")) * 100),
            2
        ).alias("late_pct"),
        round_(avg(col("delivery_days")), 1).alias("avg_delivery_days")
    ).sort(col("month"))
    
    print(f"✓ Analyzed {trends.count()} time periods")
    return trends


# ============================================================================
# 11. MAIN PIPELINE EXECUTION
# ============================================================================

def run_full_pipeline(session):
    """
    Execute the complete sentiment analysis pipeline
    """
    print("=" * 70)
    print("SENTIMENT ANALYSIS PIPELINE - CLEAN_REVIEWS")
    print("=" * 70)
    
    results = {}
    
    # Step 1: Enrich reviews with sentiment
    print("\n[1/9] Enriching reviews with sentiment...")
    results['enriched_reviews'] = enrich_reviews_with_sentiment(session)
    
    # Step 2: Analyze by product
    print("\n[2/9] Analyzing sentiment by product...")
    results['product_analysis'] = analyze_sentiment_by_product(session)
    
    # Step 3: Analyze by region
    print("\n[3/9] Analyzing sentiment by region...")
    results['region_analysis'] = analyze_sentiment_by_region(session)
    
    # Step 4: Analyze by carrier
    print("\n[4/9] Analyzing sentiment by carrier...")
    results['carrier_analysis'] = analyze_sentiment_by_carrier(session)
    
    # Step 5: Correlate with delivery delays
    print("\n[5/9] Analyzing delivery delay correlation...")
    results['correlation_stats'], results['delay_df'] = correlate_sentiment_with_delays(session)
    
    # Step 6: Delivery time bucket analysis
    print("\n[6/9] Analyzing by delivery time...")
    results['time_analysis'] = analyze_by_delivery_time(session)
    
    # Step 7: Identify problem areas
    print("\n[7/9] Identifying problem areas...")
    results['problem_areas'] = identify_problem_areas(session)
    
    # Step 8: Sentiment trends
    print("\n[8/9] Analyzing sentiment trends...")
    results['trends'] = analyze_sentiment_trends(session)
    
    # Step 9: Create dashboard view
    print("\n[9/9] Creating dashboard view...")
    results['dashboard_view'] = create_dashboard_view(session)
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE ✓")
    print("=" * 70)
    
    return results


def display_summary(results):
    """
    Display key insights from the pipeline results
    """
    print("\n" + "=" * 70)
    print("PIPELINE SUMMARY")
    print("=" * 70)
    
    print("\n--- Top 5 Products by Sentiment ---")
    results['product_analysis'].show(5)
    
    print("\n--- Region Analysis ---")
    results['region_analysis'].show()
    
    print("\n--- Carrier Performance ---")
    results['carrier_analysis'].show()
    
    print("\n--- Correlation Statistics ---")
    results['correlation_stats'].show()
    
    print("\n--- Delivery Time Impact ---")
    results['time_analysis'].show()
    
    print("\n--- Top 10 Problem Areas ---")
    results['problem_areas'].show(10)


# ============================================================================
# 12. QUICK START FUNCTIONS FOR NOTEBOOK USE
# ============================================================================

def quick_enrich():
    """Quick function to enrich reviews - use in notebook"""
    return enrich_reviews_with_sentiment(session)

def quick_analysis():
    """Quick function to run all analyses - use in notebook"""
    results = run_full_pipeline(session)
    display_summary(results)
    return results

# ============================================================================
# 13. EXAMPLE USAGE IN SNOWFLAKE NOTEBOOK
# ============================================================================

# Simply run these commands in your notebook:

# Option 1: Run full pipeline
# results = run_full_pipeline(session)
# display_summary(results)

# Option 2: Run individual analyses
# product_analysis = analyze_sentiment_by_product(session)
# product_analysis.show()

# Option 3: Use quick start functions
# results = quick_analysis()

# Access specific results
# results['product_analysis'].show()
# results['correlation_stats'].show()
# results['problem_areas'].show(20)

In [ ]:
# The session is already initialized at the top of the code
# Just run the functions directly:

# Option 1: Run the complete pipeline
results = run_full_pipeline(session)
display_summary(results)

# Option 2: Run individual analyses
product_analysis = analyze_sentiment_by_product(session)
product_analysis.show()

region_analysis = analyze_sentiment_by_region(session)
region_analysis.show()

# Option 3: Use the quick start helper
results = quick_analysis()  # Runs everything and displays summary

# Access specific results
results['product_analysis'].show(10)
results['correlation_stats'].show()
results['problem_areas'].show(20)

In [ ]:
"""
Visualization Code for Sentiment Analysis in Snowflake Notebooks
Uses Plotly, Matplotlib, and Seaborn for comprehensive visualizations
"""

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Set style for matplotlib/seaborn
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================================================
# 1. SENTIMENT DISTRIBUTION VISUALIZATIONS
# ============================================================================

def plot_sentiment_distribution():
    """Plot overall sentiment score distribution"""
    # Get data
    query = """
    SELECT 
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Create histogram with Plotly
    fig = px.histogram(
        df, 
        x='SENTIMENT_SCORE',
        nbins=50,
        title='Distribution of Sentiment Scores',
        labels={'SENTIMENT_SCORE': 'Sentiment Score', 'count': 'Number of Reviews'},
        color_discrete_sequence=['#636EFA']
    )
    
    # Add vertical lines for thresholds
    fig.add_vline(x=0.5, line_dash="dash", line_color="green", annotation_text="Positive Threshold")
    fig.add_vline(x=-0.5, line_dash="dash", line_color="red", annotation_text="Negative Threshold")
    fig.add_vline(x=0, line_dash="dot", line_color="gray", annotation_text="Neutral")
    
    fig.update_layout(showlegend=False, height=500)
    fig.show()


def plot_sentiment_categories():
    """Plot sentiment category breakdown (pie chart)"""
    query = """
    SELECT 
        CASE 
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) >= 0.5 THEN 'Positive'
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 'Negative'
            ELSE 'Neutral'
        END AS sentiment_label,
        COUNT(*) AS count
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY sentiment_label
    """
    df = session.sql(query).to_pandas()
    
    # Create pie chart
    fig = px.pie(
        df,
        values='COUNT',
        names='SENTIMENT_LABEL',
        title='Sentiment Category Distribution',
        color='SENTIMENT_LABEL',
        color_discrete_map={
            'Positive': '#00CC96',
            'Neutral': '#FFA15A',
            'Negative': '#EF553B'
        }
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(height=500)
    fig.show()


# ============================================================================
# 2. SENTIMENT BY PRODUCT
# ============================================================================

def plot_sentiment_by_product(top_n=10):
    """Plot average sentiment by product"""
    query = f"""
    SELECT 
        product,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY product
    ORDER BY total_reviews DESC
    LIMIT {top_n}
    """
    df = session.sql(query).to_pandas()
    
    # Create bar chart with dual axis
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Bar(
            x=df['PRODUCT'],
            y=df['AVG_SENTIMENT'],
            name='Avg Sentiment',
            marker_color='#636EFA'
        ),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Scatter(
            x=df['PRODUCT'],
            y=df['LATE_PCT'],
            name='Late Delivery %',
            mode='lines+markers',
            marker=dict(size=10, color='#EF553B'),
            line=dict(width=3)
        ),
        secondary_y=True
    )
    
    fig.update_xaxes(title_text="Product")
    fig.update_yaxes(title_text="Average Sentiment Score", secondary_y=False)
    fig.update_yaxes(title_text="Late Delivery %", secondary_y=True)
    
    fig.update_layout(
        title=f'Top {top_n} Products: Sentiment vs Late Deliveries',
        height=600,
        hovermode='x unified'
    )
    
    fig.show()


def plot_product_sentiment_heatmap():
    """Create heatmap of product sentiment across regions"""
    query = """
    SELECT 
        product,
        region,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY product, region
    """
    df = session.sql(query).to_pandas()
    
    # Pivot for heatmap
    pivot_df = df.pivot(index='PRODUCT', columns='REGION', values='AVG_SENTIMENT')
    
    # Create heatmap
    fig = px.imshow(
        pivot_df,
        title='Sentiment Heatmap: Products vs Regions',
        labels=dict(x="Region", y="Product", color="Avg Sentiment"),
        color_continuous_scale='RdYlGn',
        aspect='auto'
    )
    
    fig.update_layout(height=600)
    fig.show()


# ============================================================================
# 3. SENTIMENT VS DELIVERY PERFORMANCE
# ============================================================================

def plot_sentiment_vs_delivery_days():
    """Scatter plot: sentiment vs delivery days"""
    query = """
    SELECT 
        delivery_days,
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score,
        late,
        product
    FROM clean_reviews
    WHERE review_text IS NOT NULL 
        AND delivery_days IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Create scatter plot
    fig = px.scatter(
        df,
        x='DELIVERY_DAYS',
        y='SENTIMENT_SCORE',
        color='LATE',
        title='Sentiment vs Delivery Days',
        labels={
            'DELIVERY_DAYS': 'Delivery Days',
            'SENTIMENT_SCORE': 'Sentiment Score',
            'LATE': 'Late Delivery'
        },
        color_discrete_map={True: '#EF553B', False: '#00CC96'},
        opacity=0.6,
        trendline='lowess'
    )
    
    fig.update_layout(height=600)
    fig.show()


def plot_late_vs_ontime_comparison():
    """Box plot comparing late vs on-time deliveries"""
    query = """
    SELECT 
        CASE WHEN late = TRUE THEN 'Late' ELSE 'On-Time' END AS delivery_status,
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Create box plot
    fig = px.box(
        df,
        x='DELIVERY_STATUS',
        y='SENTIMENT_SCORE',
        color='DELIVERY_STATUS',
        title='Sentiment Distribution: Late vs On-Time Deliveries',
        labels={'SENTIMENT_SCORE': 'Sentiment Score', 'DELIVERY_STATUS': 'Delivery Status'},
        color_discrete_map={'Late': '#EF553B', 'On-Time': '#00CC96'}
    )
    
    fig.update_layout(height=600, showlegend=False)
    fig.show()


def plot_delivery_time_buckets():
    """Bar chart showing sentiment by delivery time buckets"""
    query = """
    SELECT 
        CASE 
            WHEN delivery_days <= 2 THEN '0-2 Days'
            WHEN delivery_days <= 4 THEN '3-4 Days'
            WHEN delivery_days <= 6 THEN '5-6 Days'
            WHEN delivery_days <= 8 THEN '7-8 Days'
            ELSE '9+ Days'
        END AS delivery_bucket,
        COUNT(*) AS review_count,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS negative_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL AND delivery_days IS NOT NULL
    GROUP BY delivery_bucket
    ORDER BY delivery_bucket
    """
    df = session.sql(query).to_pandas()
    
    # Create grouped bar chart
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Bar(
            x=df['DELIVERY_BUCKET'],
            y=df['AVG_SENTIMENT'],
            name='Avg Sentiment',
            marker_color='#636EFA'
        ),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Bar(
            x=df['DELIVERY_BUCKET'],
            y=df['NEGATIVE_PCT'],
            name='Negative %',
            marker_color='#EF553B',
            opacity=0.7
        ),
        secondary_y=True
    )
    
    fig.update_xaxes(title_text="Delivery Time")
    fig.update_yaxes(title_text="Average Sentiment Score", secondary_y=False)
    fig.update_yaxes(title_text="Negative Reviews %", secondary_y=True)
    
    fig.update_layout(
        title='Sentiment and Negative Reviews by Delivery Time',
        height=600,
        barmode='overlay'
    )
    
    fig.show()


# ============================================================================
# 4. REGIONAL ANALYSIS
# ============================================================================

def plot_sentiment_by_region():
    """Horizontal bar chart of sentiment by region"""
    query = """
    SELECT 
        region,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY region
    ORDER BY avg_sentiment DESC
    """
    df = session.sql(query).to_pandas()
    
    # Create horizontal bar chart
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=df['REGION'],
        x=df['AVG_SENTIMENT'],
        orientation='h',
        marker=dict(
            color=df['AVG_SENTIMENT'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="Sentiment")
        ),
        text=df['AVG_SENTIMENT'],
        textposition='auto',
        hovertemplate='<b>%{y}</b><br>Sentiment: %{x:.3f}<br>Late %: %{customdata[0]:.1f}%<extra></extra>',
        customdata=df[['LATE_PCT']]
    ))
    
    fig.update_layout(
        title='Average Sentiment by Region',
        xaxis_title='Average Sentiment Score',
        yaxis_title='Region',
        height=500
    )
    
    fig.show()


def plot_regional_map():
    """Scatter map showing sentiment by location"""
    query = """
    SELECT 
        region,
        latitude,
        longitude,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        COUNT(*) AS review_count
    FROM clean_reviews
    WHERE review_text IS NOT NULL 
        AND latitude IS NOT NULL 
        AND longitude IS NOT NULL
    GROUP BY region, latitude, longitude
    """
    df = session.sql(query).to_pandas()
    
    # Create scatter map
    fig = px.scatter_geo(
        df,
        lat='LATITUDE',
        lon='LONGITUDE',
        color='AVG_SENTIMENT',
        size='REVIEW_COUNT',
        hover_name='REGION',
        title='Sentiment by Geographic Location',
        color_continuous_scale='RdYlGn',
        size_max=30
    )
    
    fig.update_geos(
        scope='usa',
        showcountries=True,
        showsubunits=True
    )
    
    fig.update_layout(height=700)
    fig.show()


# ============================================================================
# 5. CARRIER PERFORMANCE
# ============================================================================

def plot_carrier_comparison():
    """Compare carriers on sentiment and delivery performance"""
    query = """
    SELECT 
        carrier,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL AND carrier IS NOT NULL
    GROUP BY carrier
    ORDER BY avg_sentiment DESC
    """
    df = session.sql(query).to_pandas()
    
    # Create grouped bar chart
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='Avg Sentiment',
        x=df['CARRIER'],
        y=df['AVG_SENTIMENT'],
        marker_color='#636EFA'
    ))
    
    fig.add_trace(go.Bar(
        name='Avg Delivery Days',
        x=df['CARRIER'],
        y=df['AVG_DELIVERY_DAYS'],
        marker_color='#FFA15A',
        yaxis='y2'
    ))
    
    fig.update_layout(
        title='Carrier Performance: Sentiment vs Delivery Time',
        xaxis_title='Carrier',
        yaxis=dict(title='Average Sentiment Score', side='left'),
        yaxis2=dict(title='Average Delivery Days', overlaying='y', side='right'),
        barmode='group',
        height=600
    )
    
    fig.show()


# ============================================================================
# 6. TIME SERIES ANALYSIS
# ============================================================================

def plot_sentiment_trends():
    """Line chart showing sentiment trends over time"""
    query = """
    SELECT 
        DATE_TRUNC('month', review_date) AS month,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) >= 0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS positive_pct,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS negative_pct,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY month
    ORDER BY month
    """
    df = session.sql(query).to_pandas()
    
    # Create multi-line chart
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Scatter(
            x=df['MONTH'],
            y=df['AVG_SENTIMENT'],
            name='Avg Sentiment',
            mode='lines+markers',
            line=dict(width=3, color='#636EFA')
        ),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Scatter(
            x=df['MONTH'],
            y=df['LATE_PCT'],
            name='Late Delivery %',
            mode='lines+markers',
            line=dict(width=2, color='#EF553B', dash='dash')
        ),
        secondary_y=True
    )
    
    fig.update_xaxes(title_text="Month")
    fig.update_yaxes(title_text="Average Sentiment Score", secondary_y=False)
    fig.update_yaxes(title_text="Late Delivery %", secondary_y=True)
    
    fig.update_layout(
        title='Sentiment and Late Delivery Trends Over Time',
        height=600,
        hovermode='x unified'
    )
    
    fig.show()


# ============================================================================
# 7. CORRELATION MATRIX
# ============================================================================

def plot_correlation_matrix():
    """Create correlation heatmap between key metrics"""
    query = """
    SELECT 
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score,
        delivery_days,
        CASE WHEN late = TRUE THEN 1 ELSE 0 END AS is_late
    FROM clean_reviews
    WHERE review_text IS NOT NULL 
        AND delivery_days IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Calculate correlations
    corr_matrix = df.corr()
    
    # Create heatmap with matplotlib
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt='.3f',
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=1,
        cbar_kws={"shrink": 0.8}
    )
    plt.title('Correlation Matrix: Sentiment vs Delivery Metrics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ============================================================================
# 8. COMPREHENSIVE DASHBOARD - ALL VISUALIZATIONS
# ============================================================================

def create_comprehensive_dashboard():
    """Run all visualizations in sequence"""
    print("=" * 70)
    print("SENTIMENT ANALYSIS DASHBOARD")
    print("=" * 70)
    
    print("\n1. Sentiment Distribution")
    plot_sentiment_distribution()
    
    print("\n2. Sentiment Categories")
    plot_sentiment_categories()
    
    print("\n3. Sentiment by Product")
    plot_sentiment_by_product()
    
    print("\n4. Late vs On-Time Comparison")
    plot_late_vs_ontime_comparison()
    
    print("\n5. Sentiment vs Delivery Days")
    plot_sentiment_vs_delivery_days()
    
    print("\n6. Delivery Time Buckets")
    plot_delivery_time_buckets()
    
    print("\n7. Sentiment by Region")
    plot_sentiment_by_region()
    
    print("\n8. Carrier Comparison")
    plot_carrier_comparison()
    
    print("\n9. Sentiment Trends Over Time")
    plot_sentiment_trends()
    
    print("\n10. Correlation Matrix")
    plot_correlation_matrix()
    
    print("\n" + "=" * 70)
    print("DASHBOARD COMPLETE ✓")
    print("=" * 70)


# ============================================================================
# USAGE EXAMPLES
# ============================================================================

# Run individual visualizations:
# plot_sentiment_distribution()
# plot_sentiment_by_product(top_n=15)
# plot_late_vs_ontime_comparison()
# plot_sentiment_by_region()
# plot_sentiment_trends()

# Or run all visualizations at once:
# create_comprehensive_dashboard()

In [ ]:
# Run individual visualizations
plot_sentiment_distribution()
plot_sentiment_by_product(top_n=10)
plot_late_vs_ontime_comparison()

# Or run ALL visualizations at once
create_comprehensive_dashboard()

In [ ]:
"""
Visualization Code for Sentiment Analysis in Snowflake Notebooks
Uses Streamlit and Altair (natively supported in Snowflake)
"""

import streamlit as st
import altair as alt
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# ============================================================================
# 1. SENTIMENT DISTRIBUTION VISUALIZATIONS
# ============================================================================

def plot_sentiment_distribution():
    """Plot overall sentiment score distribution"""
    st.subheader("📊 Sentiment Score Distribution")
    
    # Get data
    query = """
    SELECT 
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Create histogram using Altair
    chart = alt.Chart(df).mark_bar().encode(
        alt.X('SENTIMENT_SCORE:Q', bin=alt.Bin(maxbins=50), title='Sentiment Score'),
        alt.Y('count()', title='Number of Reviews'),
        tooltip=['count()']
    ).properties(
        width=700,
        height=400,
        title='Distribution of Sentiment Scores'
    )
    
    st.altair_chart(chart, use_container_width=True)
    
    # Show statistics
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Mean Sentiment", f"{df['SENTIMENT_SCORE'].mean():.3f}")
    with col2:
        st.metric("Median Sentiment", f"{df['SENTIMENT_SCORE'].median():.3f}")
    with col3:
        st.metric("Std Dev", f"{df['SENTIMENT_SCORE'].std():.3f}")


def plot_sentiment_categories():
    """Plot sentiment category breakdown"""
    st.subheader("🎯 Sentiment Category Breakdown")
    
    query = """
    SELECT 
        CASE 
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) >= 0.5 THEN 'Positive'
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 'Negative'
            ELSE 'Neutral'
        END AS sentiment_label,
        COUNT(*) AS count
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY sentiment_label
    """
    df = session.sql(query).to_pandas()
    
    # Create pie chart using Altair
    chart = alt.Chart(df).mark_arc().encode(
        theta=alt.Theta('COUNT:Q'),
        color=alt.Color('SENTIMENT_LABEL:N', 
                       scale=alt.Scale(
                           domain=['Positive', 'Neutral', 'Negative'],
                           range=['#00CC96', '#FFA15A', '#EF553B']
                       ),
                       legend=alt.Legend(title="Sentiment")),
        tooltip=['SENTIMENT_LABEL', 'COUNT']
    ).properties(
        width=400,
        height=400,
        title='Sentiment Categories'
    )
    
    st.altair_chart(chart, use_container_width=True)
    
    # Show percentages
    total = df['COUNT'].sum()
    for _, row in df.iterrows():
        pct = (row['COUNT'] / total) * 100
        st.write(f"**{row['SENTIMENT_LABEL']}**: {row['COUNT']:,} ({pct:.1f}%)")


# ============================================================================
# 2. SENTIMENT BY PRODUCT
# ============================================================================

def plot_sentiment_by_product(top_n=10):
    """Plot average sentiment by product"""
    st.subheader(f"🛍️ Top {top_n} Products by Sentiment")
    
    query = f"""
    SELECT 
        product,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY product
    ORDER BY total_reviews DESC
    LIMIT {top_n}
    """
    df = session.sql(query).to_pandas()
    
    # Create bar chart
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('AVG_SENTIMENT:Q', title='Average Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        y=alt.Y('PRODUCT:N', sort='-x', title='Product'),
        color=alt.Color('AVG_SENTIMENT:Q', 
                       scale=alt.Scale(scheme='redyellowgreen', domain=[-1, 1]),
                       legend=None),
        tooltip=['PRODUCT', 'AVG_SENTIMENT', 'TOTAL_REVIEWS', 'LATE_PCT']
    ).properties(
        width=700,
        height=400,
        title=f'Average Sentiment by Product (Top {top_n})'
    )
    
    st.altair_chart(chart, use_container_width=True)
    
    # Show data table
    with st.expander("📋 View Data Table"):
        st.dataframe(df, use_container_width=True)


# ============================================================================
# 3. SENTIMENT VS DELIVERY PERFORMANCE
# ============================================================================

def plot_late_vs_ontime_comparison():
    """Compare late vs on-time deliveries"""
    st.subheader("⏰ Late vs On-Time Delivery Comparison")
    
    query = """
    SELECT 
        CASE WHEN late = TRUE THEN 'Late' ELSE 'On-Time' END AS delivery_status,
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    # Create box plot
    chart = alt.Chart(df).mark_boxplot(size=50).encode(
        x=alt.X('DELIVERY_STATUS:N', title='Delivery Status'),
        y=alt.Y('SENTIMENT_SCORE:Q', title='Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        color=alt.Color('DELIVERY_STATUS:N',
                       scale=alt.Scale(
                           domain=['On-Time', 'Late'],
                           range=['#00CC96', '#EF553B']
                       ),
                       legend=None)
    ).properties(
        width=500,
        height=400,
        title='Sentiment Distribution: Late vs On-Time'
    )
    
    st.altair_chart(chart, use_container_width=True)
    
    # Show statistics
    stats_query = """
    SELECT 
        CASE WHEN late = TRUE THEN 'Late' ELSE 'On-Time' END AS delivery_status,
        COUNT(*) AS count,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(MIN(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS min_sentiment,
        ROUND(MAX(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS max_sentiment
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY delivery_status
    """
    stats_df = session.sql(stats_query).to_pandas()
    st.dataframe(stats_df, use_container_width=True)


def plot_sentiment_vs_delivery_days():
    """Scatter plot: sentiment vs delivery days"""
    st.subheader("📦 Sentiment vs Delivery Days")
    
    query = """
    SELECT 
        delivery_days,
        SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score,
        late
    FROM clean_reviews
    WHERE review_text IS NOT NULL 
        AND delivery_days IS NOT NULL
    LIMIT 1000
    """
    df = session.sql(query).to_pandas()
    
    # Create scatter plot
    chart = alt.Chart(df).mark_circle(size=60, opacity=0.5).encode(
        x=alt.X('DELIVERY_DAYS:Q', title='Delivery Days'),
        y=alt.Y('SENTIMENT_SCORE:Q', title='Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        color=alt.Color('LATE:N',
                       scale=alt.Scale(
                           domain=[False, True],
                           range=['#00CC96', '#EF553B']
                       ),
                       legend=alt.Legend(title="Late Delivery")),
        tooltip=['DELIVERY_DAYS', 'SENTIMENT_SCORE', 'LATE']
    ).properties(
        width=700,
        height=400,
        title='Sentiment vs Delivery Days (Sample of 1000 reviews)'
    )
    
    st.altair_chart(chart, use_container_width=True)


def plot_delivery_time_buckets():
    """Bar chart showing sentiment by delivery time buckets"""
    st.subheader("⏱️ Sentiment by Delivery Time Buckets")
    
    query = """
    SELECT 
        CASE 
            WHEN delivery_days <= 2 THEN '0-2 Days'
            WHEN delivery_days <= 4 THEN '3-4 Days'
            WHEN delivery_days <= 6 THEN '5-6 Days'
            WHEN delivery_days <= 8 THEN '7-8 Days'
            ELSE '9+ Days'
        END AS delivery_bucket,
        COUNT(*) AS review_count,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS negative_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL AND delivery_days IS NOT NULL
    GROUP BY delivery_bucket
    """
    df = session.sql(query).to_pandas()
    
    # Ensure proper ordering
    order = ['0-2 Days', '3-4 Days', '5-6 Days', '7-8 Days', '9+ Days']
    
    # Create bar chart
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('DELIVERY_BUCKET:N', sort=order, title='Delivery Time'),
        y=alt.Y('AVG_SENTIMENT:Q', title='Average Sentiment Score'),
        color=alt.Color('AVG_SENTIMENT:Q', 
                       scale=alt.Scale(scheme='redyellowgreen', domain=[-1, 1]),
                       legend=None),
        tooltip=['DELIVERY_BUCKET', 'AVG_SENTIMENT', 'REVIEW_COUNT', 'NEGATIVE_PCT']
    ).properties(
        width=700,
        height=400,
        title='Average Sentiment by Delivery Time'
    )
    
    st.altair_chart(chart, use_container_width=True)
    st.dataframe(df, use_container_width=True)


# ============================================================================
# 4. REGIONAL ANALYSIS
# ============================================================================

def plot_sentiment_by_region():
    """Bar chart of sentiment by region"""
    st.subheader("🌎 Sentiment by Region")
    
    query = """
    SELECT 
        region,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY region
    ORDER BY avg_sentiment DESC
    """
    df = session.sql(query).to_pandas()
    
    # Create horizontal bar chart
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('AVG_SENTIMENT:Q', title='Average Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        y=alt.Y('REGION:N', sort='-x', title='Region'),
        color=alt.Color('AVG_SENTIMENT:Q', 
                       scale=alt.Scale(scheme='redyellowgreen', domain=[-1, 1]),
                       legend=None),
        tooltip=['REGION', 'AVG_SENTIMENT', 'TOTAL_REVIEWS', 'AVG_DELIVERY_DAYS', 'LATE_PCT']
    ).properties(
        width=700,
        height=400,
        title='Average Sentiment by Region'
    )
    
    st.altair_chart(chart, use_container_width=True)
    
    with st.expander("📋 View Regional Data"):
        st.dataframe(df, use_container_width=True)


# ============================================================================
# 5. CARRIER PERFORMANCE
# ============================================================================

def plot_carrier_comparison():
    """Compare carriers on sentiment and delivery performance"""
    st.subheader("🚚 Carrier Performance Comparison")
    
    query = """
    SELECT 
        carrier,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL AND carrier IS NOT NULL
    GROUP BY carrier
    ORDER BY avg_sentiment DESC
    """
    df = session.sql(query).to_pandas()
    
    # Create bar chart
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('AVG_SENTIMENT:Q', title='Average Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        y=alt.Y('CARRIER:N', sort='-x', title='Carrier'),
        color=alt.Color('AVG_SENTIMENT:Q', 
                       scale=alt.Scale(scheme='redyellowgreen', domain=[-1, 1]),
                       legend=None),
        tooltip=['CARRIER', 'AVG_SENTIMENT', 'TOTAL_REVIEWS', 'AVG_DELIVERY_DAYS', 'LATE_PCT']
    ).properties(
        width=700,
        height=400,
        title='Carrier Performance: Average Sentiment'
    )
    
    st.altair_chart(chart, use_container_width=True)
    st.dataframe(df, use_container_width=True)


# ============================================================================
# 6. TIME SERIES ANALYSIS
# ============================================================================

def plot_sentiment_trends():
    """Line chart showing sentiment trends over time"""
    st.subheader("📈 Sentiment Trends Over Time")
    
    query = """
    SELECT 
        DATE_TRUNC('month', review_date) AS month,
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    GROUP BY month
    ORDER BY month
    """
    df = session.sql(query).to_pandas()
    
    # Create line chart for sentiment
    sentiment_chart = alt.Chart(df).mark_line(point=True, strokeWidth=3).encode(
        x=alt.X('MONTH:T', title='Month'),
        y=alt.Y('AVG_SENTIMENT:Q', title='Average Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
        tooltip=['MONTH', 'AVG_SENTIMENT', 'TOTAL_REVIEWS', 'LATE_PCT']
    ).properties(
        width=700,
        height=400,
        title='Average Sentiment Over Time'
    )
    
    st.altair_chart(sentiment_chart, use_container_width=True)
    
    # Create line chart for late percentage
    late_chart = alt.Chart(df).mark_line(point=True, strokeWidth=3, color='#EF553B').encode(
        x=alt.X('MONTH:T', title='Month'),
        y=alt.Y('LATE_PCT:Q', title='Late Delivery %'),
        tooltip=['MONTH', 'LATE_PCT', 'TOTAL_REVIEWS']
    ).properties(
        width=700,
        height=300,
        title='Late Delivery % Over Time'
    )
    
    st.altair_chart(late_chart, use_container_width=True)


# ============================================================================
# 7. CORRELATION ANALYSIS
# ============================================================================

def show_correlation_stats():
    """Display correlation statistics"""
    st.subheader("🔗 Correlation Analysis")
    
    query = """
    SELECT 
        CORR(SNOWFLAKE.CORTEX.SENTIMENT(review_text), delivery_days) AS sentiment_delivery_correlation,
        AVG(CASE WHEN late = TRUE THEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) END) AS avg_sentiment_late,
        AVG(CASE WHEN late = FALSE THEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) END) AS avg_sentiment_ontime
    FROM clean_reviews
    WHERE review_text IS NOT NULL AND delivery_days IS NOT NULL
    """
    df = session.sql(query).to_pandas()
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric(
            "Sentiment-Delivery Correlation", 
            f"{df['SENTIMENT_DELIVERY_CORRELATION'].iloc[0]:.3f}",
            help="Correlation between sentiment score and delivery days"
        )
    
    with col2:
        st.metric(
            "Avg Sentiment (Late)", 
            f"{df['AVG_SENTIMENT_LATE'].iloc[0]:.3f}"
        )
    
    with col3:
        st.metric(
            "Avg Sentiment (On-Time)", 
            f"{df['AVG_SENTIMENT_ONTIME'].iloc[0]:.3f}"
        )
    
    sentiment_diff = df['AVG_SENTIMENT_ONTIME'].iloc[0] - df['AVG_SENTIMENT_LATE'].iloc[0]
    st.info(f"💡 **Insight**: On-time deliveries have {sentiment_diff:.3f} higher sentiment on average")


# ============================================================================
# 8. COMPREHENSIVE DASHBOARD
# ============================================================================

def create_comprehensive_dashboard():
    """Run all visualizations in sequence"""
    st.title("🎯 Sentiment Analysis Dashboard")
    st.markdown("---")
    
    # Overview metrics
    st.header("📊 Overview")
    overview_query = """
    SELECT 
        COUNT(*) AS total_reviews,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(review_text)), 3) AS avg_sentiment,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) >= 0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS positive_pct,
        ROUND(100.0 * SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_text) <= -0.5 THEN 1 ELSE 0 END) / COUNT(*), 2) AS negative_pct,
        ROUND(100.0 * SUM(CASE WHEN late = TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
    FROM clean_reviews
    WHERE review_text IS NOT NULL
    """
    overview_df = session.sql(overview_query).to_pandas()
    
    col1, col2, col3, col4, col5 = st.columns(5)
    with col1:
        st.metric("Total Reviews", f"{overview_df['TOTAL_REVIEWS'].iloc[0]:,}")
    with col2:
        st.metric("Avg Sentiment", f"{overview_df['AVG_SENTIMENT'].iloc[0]:.3f}")
    with col3:
        st.metric("Positive %", f"{overview_df['POSITIVE_PCT'].iloc[0]:.1f}%")
    with col4:
        st.metric("Negative %", f"{overview_df['NEGATIVE_PCT'].iloc[0]:.1f}%")
    with col5:
        st.metric("Late Delivery %", f"{overview_df['LATE_PCT'].iloc[0]:.1f}%")
    
    st.markdown("---")
    
    # All visualizations
    plot_sentiment_distribution()
    st.markdown("---")
    
    plot_sentiment_categories()
    st.markdown("---")
    
    plot_sentiment_by_product()
    st.markdown("---")
    
    plot_late_vs_ontime_comparison()
    st.markdown("---")
    
    plot_delivery_time_buckets()
    st.markdown("---")
    
    plot_sentiment_by_region()
    st.markdown("---")
    
    plot_carrier_comparison()
    st.markdown("---")
    
    plot_sentiment_trends()
    st.markdown("---")
    
    show_correlation_stats()
    
    st.success("✅ Dashboard Complete!")


# ============================================================================
# USAGE
# ============================================================================

# Run the comprehensive dashboard
create_comprehensive_dashboard()

# Or run individual visualizations:
# plot_sentiment_distribution()
# plot_sentiment_by_product(top_n=15)
# plot_late_vs_ontime_comparison()
# plot_sentiment_by_region()
# plot_sentiment_trends()

In [ ]:
# After running the code, call individual functions:
plot_sentiment_distribution()
plot_sentiment_by_product(top_n=15)
plot_late_vs_ontime_comparison()